In [8]:
from typing import TypedDict, Annotated

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

from langchain_core.messages import BaseMessage
from langchain_core.tools import tool

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama

from sentence_transformers import CrossEncoder

In [9]:
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100
K = 3
CANDIDATE_K = 10

PDF_PATH = r"C:\Users\bjit\Downloads\attention.pdf"

EMBEDDING_MODEL = "nomic-embed-text"
LLM_MODEL = "qwen2.5:3b"

RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

In [10]:
# LOAD EMBEDDINGS

embeddings = OllamaEmbeddings(
    model=EMBEDDING_MODEL
)

In [11]:
# ============================================================
# 1. LOAD PDF
# ============================================================

loader = PyPDFLoader(PDF_PATH)

docs = loader.load()

print(f"Number of pages loaded: {len(docs)}")

Number of pages loaded: 15


In [12]:
# Inspect the first page

print(docs[0].page_content[:2000])

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Exper

In [13]:
print(docs[0].metadata)

{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'C:\\Users\\bjit\\Downloads\\attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}


In [ ]:
# SEMANTIC CHUNKING

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

chunks = splitter.split_documents(docs)

print(f"Created {len(chunks)} chunks")

Created 103 semantic chunks


In [15]:
# Inspect the first chunk

print(chunks[0].page_content)

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu


In [16]:
print(chunks[0].metadata)

{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'C:\\Users\\bjit\\Downloads\\attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}


In [17]:
for i, chunk in enumerate(chunks[:5]):
    print(f"\n--- Chunk {i} ---")
    print(chunk.page_content[:500])


--- Chunk 0 ---
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu

--- Chunk 1 ---
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention

--- Chunk 2 ---
performing models also conn

In [18]:
# ============================================================
# 3. CREATE EMBEDDINGS
# ============================================================

# LOAD EMBEDDINGS

embeddings = OllamaEmbeddings(
    model=EMBEDDING_MODEL
)

print(f"Embedding model: {EMBEDDING_MODEL}")

Embedding model: nomic-embed-text


In [19]:
# ============================================================
# 4. CREATE FAISS VECTOR STORE
# ============================================================

vector_store = FAISS.from_documents(
    chunks,
    embeddings
)

print("FAISS vector store created.")

FAISS vector store created.


In [20]:
# SPARSE RETRIEVER

bm25_retriever = BM25Retriever.from_documents(
    chunks,
    k=CANDIDATE_K
)

In [21]:
# RERANKER

reranker = CrossEncoder(
    RERANKER_MODEL
)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [24]:
# ============================================================
# 6. LOAD QWEN LLM
# ============================================================

llm = ChatOllama(
    model=LLM_MODEL
)

print(f"LLM model: {LLM_MODEL}")

LLM model: qwen2.5:3b


In [25]:
# ============================================================
# 7. DEFINE RAG STATE
# ============================================================

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [26]:
def retrieve_documents(query):

    # 1. Query rephrasing
    rewritten = llm.invoke(
        f"""
Rewrite the user's question into a clear search query
for retrieving relevant documents.

User question:
{query}

Return only the rewritten query.
"""
    ).content.strip()

    # 2. HyDE
    hypothetical_doc = llm.invoke(
        f"""
Write a short hypothetical passage that would answer
this question. This passage will be used only for
semantic document retrieval.

Question:
{rewritten}
"""
    ).content.strip()

    # 3. Dense retrieval using HyDE
    dense_docs = vector_store.similarity_search(
        hypothetical_doc,
        k=CANDIDATE_K
    )

    # 4. Sparse retrieval using rewritten query
    sparse_docs = bm25_retriever.invoke(
        rewritten
    )

    # 5. Combine and remove duplicates
    candidates = []

    seen = set()

    for doc in dense_docs + sparse_docs:

        text = doc.page_content

        if text not in seen:
            seen.add(text)
            candidates.append(doc)

    # 6. Cross-encoder reranking
    pairs = [
        (rewritten, doc.page_content)
        for doc in candidates
    ]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(scores, candidates),
        key=lambda x: x[0],
        reverse=True
    )

    return ranked[:K]

In [27]:
@tool
def rag_tool(question: str) -> str:
    """
    Search the currently loaded document collection.

    Use this tool when the user's question may require
    information from the loaded documents.

    Do not use it for casual conversation, simple arithmetic,
    or questions that clearly do not require document information.
    """

    results = retrieve_documents(question)

    output = []

    for i, (score, doc) in enumerate(results, start=1):

        output.append(
            f"""
Candidate {i}
Reranker score: {score:.4f}
Source: {doc.metadata.get("source", "unknown")}
Page: {doc.metadata.get("page", "unknown")}

{doc.page_content}
"""
        )

    return "\n".join(output)

In [28]:
tools = [rag_tool]

llm_with_tools = llm.bind_tools(tools)

In [29]:
from langchain_core.messages import SystemMessage

In [30]:
SYSTEM_PROMPT = """
You are a helpful chatbot with access to a document retrieval tool.

Decide whether the user's question requires information from
the currently loaded documents.

If the answer may be found in the documents, use the retrieval
tool before answering.

If the question can be answered without the documents,
answer normally without using the tool.

Do not use the tool for casual conversation or simple questions
that clearly do not need document information.
"""


def chat_node(state: ChatState):

    messages = [
        SystemMessage(content=SYSTEM_PROMPT)
    ] + state["messages"]

    response = llm_with_tools.invoke(messages)

    return {
        "messages": [response]
    }

In [31]:
tool_node = ToolNode(tools)

graph = StateGraph(ChatState)

graph.add_node("chat_node", chat_node)
graph.add_node("tools", tool_node)

graph.add_edge(
    START,
    "chat_node"
)

graph.add_conditional_edges(
    "chat_node",
    tools_condition
)

graph.add_edge(
    "tools",
    "chat_node"
)

chatbot = graph.compile()

In [34]:
from langchain_core.messages import HumanMessage, ToolMessage, AIMessage

while True:

    user_input = input("\nYou: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Goodbye!")
        break

    result = chatbot.invoke({
        "messages": [
            HumanMessage(content=user_input)
        ]
    })

    rag_used = False

    print(f"\n👤 User: {user_input}")

    for message in result["messages"]:

        if isinstance(message, ToolMessage):

            rag_used = True

            print("\n🔎 RAG: USED")
            print("Top-K candidates:")
            print("-" * 60)
            print(message.content)
            print("-" * 60)

    for message in reversed(result["messages"]):

        if isinstance(message, AIMessage) and message.content:

            print("\n🤖 AI:", message.content)
            break

    if not rag_used:
        print("\n🔎 RAG: NOT USED")


👤 User: hi, i am tasrif

🤖 AI: Hello Tasrif! How can I assist you today?

🔎 RAG: NOT USED

👤 User: what is the capital of bangladesh

🔎 RAG: USED
Top-K candidates:
------------------------------------------------------------

Candidate 1
Reranker score: -11.0580
Source: C:\Users\bjit\Downloads\attention.pdf
Page: 6

orO(logk(n)) in the case of dilated convolutions [ 18], increasing the length of the longest paths
between any two positions in the network. Convolutional layers are generally more expensive than
recurrent layers, by a factor of k. Separable convolutions [ 6], however, decrease the complexity
considerably, toO(k·n·d +n·d2). Even with k = n, however, the complexity of a separable
convolution is equal to the combination of a self-attention layer and a point-wise feed-forward layer,


Candidate 2
Reranker score: -11.0689
Source: C:\Users\bjit\Downloads\attention.pdf
Page: 1

described in section 3.2.
Self-attention, sometimes called intra-attention is an attention mechanism r